# Table of Contents

1. [Load Project](##Load-Project) 
2. [Calculate Step Length and Turn Angle](#Calculate-Step-Length-and-Turn-Angle)
    1. [Visualize Step and Angle Distribution](#Visualize-Step-and-Angle-Distribution)
3. [Run CoMPASS Level 1](#Run-CoMPASS-Level-1)
    1. [Check Results](#Check-results)
    2. [Re-attach stationary frames](#Re-attach-stationary-frames)
4. [Export HMM results](#Export-HMM-results---model-and-data)

# Load Project

In [ ]:
from pathlib import Path
import pandas as pd
from compass_labyrinth import load_project


project_path = Path(r'PATH\TO\PROJECT\compass_analysis')

# Import config and metadata
config, cohort_metadata = load_project(project_path=project_path)

# Import all sessions combined pose-estimation CSV
df_all_csv = pd.read_csv(project_path / "csvs" / "combined" / "Preprocessed_combined_file_exclusions.csv")

df_all_csv

# Calculate Step Length and Turn Angle

In [ ]:
from compass_labyrinth.compass.level_1 import prep_data


# Run prep
final_df = prep_data(
    data=df_all_csv,
    type="UTM",                         # or "LL" if lon/lat in coordNames
    coordNames=("x", "y"),
    covNames=[],  # example covariates if present
    centers=None,                       # e.g., np.array([[0,0], [10,10]])
    centroids=None,                     # e.g., {"goal": goal_df_with_time}
    angleCovs=None,
    altCoordNames=None
)

print("\nSummary: Step")
print(final_df["step"].describe())

print("\nSummary: Angle (radians)")
print(final_df["angle"].describe())

### Visualize Step and Angle Distribution

In [ ]:
from compass_labyrinth.compass.level_1 import plot_step_and_angle_distributions

plot_step_and_angle_distributions(
    config=config,
    df=final_df,
)

# Run CoMPASS Level-1

Goal                               	R (momentuHMM)	                                  Python (SciPy analogue)
___________________________________________________________________________________________________________________________
Gradient-based optimization	      "BFGS", "L-BFGS-B"	                                 "BFGS", "L-BFGS-B"
Derivative-free (simplex)	         "Nelder-Mead"                                      	"Nelder-Mead"
Derivative-free (Newton-like)	         "nlm"                                              	"Powell

In [ ]:
from compass_labyrinth.compass.level_1 import fit_best_hmm
import numpy as np


# Frames with step < 1.0 have essentially no real movement, so their turning
# angle is dominated by tracking noise rather than real behavior -- this was
# destabilizing the von Mises kappa fit for the slow-movement state (kappa
# came out negative, an invalid concentration value, failing the behavioral
# constraint check on every one of 10 test repetitions). Filtering them out
# fixed it (10/10 passed in testing).
final_df_filtered = final_df[final_df["step"] >= 1.0].copy()

# final_df_filtered must have: 'session', 'step' (>0), 'angle' (radians)
res = fit_best_hmm(
    preproc_df=final_df_filtered,
    n_states=2,
    n_repetitions=10,
    opt_methods=["L-BFGS-B"],
    max_iter=50,
    use_abs_angle=(False,),          # True => |angle|~Gamma ; False => angle~VM
    stationary_flag="auto",
    use_data_driven_ranges=True,
    angle_mean_biased=(np.pi/2, 0.0),     # used for VM branch
    session_col="session",
    seed=123,
    enforce_behavioral_constraints=True,
    show_progress=True
)

### Check results

- State 1 = low step + high turn angle
- State 2 = high step + low turn angle

In [ ]:
from compass_labyrinth.compass.level_1 import print_hmm_summary


print_hmm_summary(res.summary, res.model)

res.data.head()          # original df + hmm_state + posteriors


In [ ]:
res.data.hmm_state.value_counts()

### Re-attach stationary frames

Frames with `step < 1.0` were excluded before fitting (their turning angle is dominated by tracking noise, not real behavior). Re-attach them here as a distinct `hmm_state = 0` ("stationary") label instead of silently dropping them, so every frame in the session gets an accounting.

In [ ]:
from compass_labyrinth.compass.level_1 import merge_stationary_frames


final_df_excluded = final_df[final_df["step"] < 1.0].copy()

data_with_stationary = merge_stationary_frames(res, final_df_excluded)

data_with_stationary.hmm_state.value_counts()

# Export HMM results - model and data

In [ ]:
res.save(config=config)

# Overwrite the saved data_with_states.csv with the complete, stationary-labeled
# version so the saved artifact accounts for every frame, not just moving ones.
data_path = Path(config["project_path_full"]) / "results" / "compass_level_1" / "data_with_states.csv"
data_with_stationary.to_csv(data_path, index=False)
print(f"Overwrote {data_path.name} with the stationary-labeled version ({len(data_with_stationary)} rows)")